# Representation Demystifies Flat Minima
## Notebook 17: Reparameterization

This notebook demonstrates the core idea behind **arXiv:2605.05209**:

> The same function can appear sharp or flat depending on how it is parameterized.

We use a tiny analytic model where the function is preserved exactly, but the measured curvature in parameter space changes under reparameterization.

## Output files

Running this notebook creates:

- `figures/17_same_function.png`
- `figures/17_parameter_loss_landscapes.png`
- `figures/17_sharpness_vs_scale.png`
- `results/17_reparameterization_results.csv`
- `results/17_reparameterization_results.json`
- `downloads/17_reparameterization_outputs.zip`

The final cell displays direct download links in Colab/Jupyter.

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()

# Works whether run from repo root or notebooks/
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FIGURES:", FIGURES)
print("RESULTS:", RESULTS)
print("DOWNLOADS:", DOWNLOADS)

## 1. Same function, different parameter coordinate

Define a simple scalar model:

\[
f(x; w) = wx
\]

We want the target function:

\[
f(x) = x
\]

so the optimum is:

\[
w^* = 1
\]

Now introduce a reparameterization:

\[
w = s u
\]

The same target function is reached at:

\[
u^* = \frac{1}{s}
\]

The function is unchanged at the optimum, but curvature with respect to the coordinate \(u\) changes with \(s\).

In [ ]:
x = np.linspace(-1, 1, 400)
y = x

scales = [0.25, 0.5, 1.0, 2.0, 4.0]

plt.figure(figsize=(8, 5))

for s in scales:
    u_star = 1 / s
    prediction = (s * u_star) * x
    plt.plot(x, prediction, label=f"s={s}, u*={u_star:.2f}")

plt.plot(x, y, linestyle="--", label="target y=x")

plt.xlabel("x")
plt.ylabel("prediction")
plt.title("Same function under different parameterizations")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "17_same_function.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 2. Loss in the original coordinate

Use mean squared error:

\[
L(w) = \mathbb{E}_x[(wx - x)^2]
\]

Since \(y=x\), this is:

\[
L(w) = \mathbb{E}_x[x^2](w-1)^2
\]

The Hessian with respect to \(w\) is:

\[
H_w = 2\mathbb{E}_x[x^2]
\]

In [ ]:
ex2 = np.mean(x**2)
hessian_w = 2 * ex2

print("E[x^2] =", ex2)
print("Hessian in w-coordinate =", hessian_w)

## 3. Loss in the reparameterized coordinate

Substitute:

\[
w = su
\]

Then:

\[
L(u) = \mathbb{E}_x[(sux - x)^2]
\]

The optimum is still functionally equivalent, but:

\[
H_u = 2s^2\mathbb{E}_x[x^2]
\]

So measured sharpness changes as \(s^2\), even though the learned function is identical.

In [ ]:
rows = []

for s in scales:
    u_star = 1 / s
    hessian_u = 2 * (s**2) * ex2

    rows.append({
        "scale_s": s,
        "u_star": u_star,
        "w_effective": s * u_star,
        "hessian_w": hessian_w,
        "hessian_u": hessian_u,
        "sharpness_ratio_hu_hw": hessian_u / hessian_w,
        "max_prediction_error": np.max(np.abs((s * u_star) * x - y))
    })

results = pd.DataFrame(rows)
results

In [ ]:
csv_path = RESULTS / "17_reparameterization_results.csv"
json_path = RESULTS / "17_reparameterization_results.json"

results.to_csv(csv_path, index=False)
results.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## 4. Parameter-space loss landscapes

Now compare the loss curves over each coordinate.

The original coordinate \(w\) has one curvature.

The reparameterized coordinate \(u\) can look much flatter or much sharper depending on \(s\).

In [ ]:
w_grid = np.linspace(-1, 3, 600)
loss_w = ex2 * (w_grid - 1)**2

plt.figure(figsize=(8, 5))
plt.plot(w_grid, loss_w, label=f"original w, H={hessian_w:.3f}")
plt.axvline(1, linestyle="--", linewidth=1)

plt.xlabel("w")
plt.ylabel("loss")
plt.title("Loss landscape in original parameter coordinate")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "17_original_coordinate_loss.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

In [ ]:
plt.figure(figsize=(9, 6))

for s in scales:
    u_star = 1 / s
    u_grid = np.linspace(u_star - 2, u_star + 2, 600)
    loss_u = ex2 * (s * u_grid - 1)**2
    hessian_u = 2 * (s**2) * ex2

    plt.plot(u_grid, loss_u, label=f"s={s}, H_u={hessian_u:.3f}")

plt.ylim(0, 2.5)
plt.xlabel("u")
plt.ylabel("loss")
plt.title("Same function target, different parameter-space sharpness")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "17_parameter_loss_landscapes.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 5. Sharpness changes while predictions do not

The model's function at the optimum is unchanged:

\[
f(x; su^*) = x
\]

But the measured Hessian changes:

\[
H_u \propto s^2
\]

This is the core demonstration:

> Same predictions. Different sharpness.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(results["scale_s"], results["hessian_u"], marker="o", label="Hessian in u-coordinate")
plt.axhline(hessian_w, linestyle="--", label="Hessian in w-coordinate")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("reparameterization scale s")
plt.ylabel("measured Hessian")
plt.title("Measured sharpness depends on parameterization")
plt.legend()
plt.grid(True, which="both")

fig_path = FIGURES / "17_sharpness_vs_scale.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 6. Notebook takeaway

This toy model does not prove that flatness is useless.

It shows something narrower and more important:

> A sharpness measurement can change even when the represented function does not.

Therefore, flatness alone is not automatically a coordinate-invariant causal explanation for generalization.

**Engineering statement:** Representation describes geometry. Computation describes function.

In [ ]:
output_files = [
    FIGURES / "17_same_function.png",
    FIGURES / "17_original_coordinate_loss.png",
    FIGURES / "17_parameter_loss_landscapes.png",
    FIGURES / "17_sharpness_vs_scale.png",
    RESULTS / "17_reparameterization_results.csv",
    RESULTS / "17_reparameterization_results.json",
]

zip_path = DOWNLOADS / "17_reparameterization_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in output_files:
        if file_path.exists():
            z.write(file_path, arcname=file_path.relative_to(ROOT))

print("Created:", zip_path)
print("\nIncluded files:")
for file_path in output_files:
    print("-", file_path.relative_to(ROOT), "exists=" + str(file_path.exists()))

## Download outputs

In Colab, run the next cell to download the ZIP directly.

In local Jupyter, open:

`downloads/17_reparameterization_outputs.zip`

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display

    display(FileLink(str(zip_path)))
    for file_path in output_files:
        if file_path.exists():
            display(FileLink(str(file_path)))